# 0 - Point it at a video and see what it finds

**Start here.** No labelling, no training, no ground truth. Give it a Mini 2
clip and look at what the detector does with it.

What this answers:
- Does it find the animals at all?
- At what confidence do they come in?
- What is it firing on that it should not be?
- Where does it lose them?

What this **cannot** answer: how accurate it is. Without labelled ground
truth there is nothing to be right or wrong against, so there is no precision
or recall here and none is reported. For a real score, label fifty frames
and use notebook 1.

That distinction matters. Looking at output and measuring accuracy are
different activities, and only the second one belongs in a writeup.

## First: does this environment actually work?

Run this before anything else. It checks Python, FFmpeg, the packages, the
GPU and the DroneMegaDetector weights all at once, so a missing binary shows
up here rather than as a puzzling traceback five cells down.

`fix=True` installs what it can -- FFmpeg via apt or conda, and the
checkpoint. It is safe to re-run.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
import doctor

doctor.check(fix=True)

In [ ]:
import json
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import detect_video as dv
import dmd
import matplotlib.pyplot as plt
from dmd import DMD_CLASSES, load_model, predict_images

HERE = Path.cwd().parent

# ---- set these two --------------------------------------------------------
VIDEO = Path.home() / "aerial-data" / "clips" / "DJI_0001.MP4"
WEIGHTS = None          # None = stock DroneMegaDetector; or a runs/.../*.pth path
# ---------------------------------------------------------------------------

SAMPLE_FPS = 2.0        # frames per second to pull out of the clip
MAX_EDGE = 1280         # downscale frames to this; DMD was trained at 1280
FLOOR = 0.05            # inference threshold - deliberately low, filter later

WORK = Path.home() / "aerial-data" / "runs" / VIDEO.stem
print("video :", VIDEO, "(exists)" if VIDEO.exists() else "(NOT FOUND - fix the path)")
print("work  :", WORK)

## What is in the clip

In [ ]:
info = dv.probe(VIDEO)
print(f"  {info['width']}x{info['height']}  {info['duration_seconds']:.1f}s  "
      f"{info['frame_rate']} fps")
print(f"  sampling at {SAMPLE_FPS} fps -> about "
      f"{int(info['duration_seconds'] * SAMPLE_FPS)} frames")

if info["width"] and info["width"] < 1280:
    print("\n  This clip is below 1280 px wide. If it came off the live RTMP")
    print("  stream rather than the memory card, that is expected - and small")
    print("  animals may simply not survive the downscaling.")

## Pull frames out

Straight FFmpeg, not `drones/video.py:prepare_video` - that caps at 60 frames
because it builds evidence bundles. This is a research tool and samples the
whole clip.

In [ ]:
frames = dv.extract(VIDEO, WORK / "frames", fps=SAMPLE_FPS, max_edge=MAX_EDGE)
print(f"{len(frames)} frames -> {WORK / 'frames'}")

## Run the detector

Predict once at a permissive floor, then re-threshold as often as you like
without re-running inference.

In [ ]:
model = load_model(WEIGHTS)
per_frame = predict_images(model, frames, threshold=FLOOR)

total = sum(len(v) for v in per_frame.values())
print(f"{total} raw detections above {FLOOR} across {len(frames)} frames")

## Where do the confidences sit?

This tells you where to put the threshold. A clean model shows two humps -
noise low down, real detections high up. One smear means it is unsure about
everything, which usually means a domain gap.

In [ ]:
figure, axes = plt.subplots(1, len(DMD_CLASSES), figsize=(14, 3.5), squeeze=False)
for column, name in enumerate(DMD_CLASSES):
    scores = [b.score for boxes in per_frame.values() for b in boxes if b.class_name == name]
    axis = axes[0][column]
    if scores:
        axis.hist(scores, bins=20, range=(0, 1), color=dmd.CLASS_COLOURS[name],
                  edgecolor="#555")
    axis.set_title(f"{name} ({len(scores)})", fontsize=10)
    axis.set_xlabel("confidence")
    axis.set_xlim(0, 1)
plt.tight_layout()
plt.show()

## Scrub through it

Drag the frame slider to walk the clip, and the confidence slider to see what
each threshold would keep. This is the cell to spend time in.

In [ ]:
try:
    from ipywidgets import FloatSlider, IntSlider, interact

    def preview(index, threshold):
        path = frames[index]
        kept = [b for b in per_frame[path] if b.score >= threshold]
        figure, axis = plt.subplots(figsize=(12, 7))
        axis.imshow(dmd.draw_boxes(path, per_frame[path], threshold=threshold))
        axis.axis("off")
        counts = ", ".join(
            f"{n}x{sum(1 for b in kept if b.class_name == n)}"
            for n in DMD_CLASSES
            if any(b.class_name == n for b in kept)
        ) or "nothing"
        axis.set_title(
            f"{path.name}  t={index / SAMPLE_FPS:.1f}s  @{threshold:.2f}  ->  {counts}",
            fontsize=11,
        )
        plt.show()

    interact(
        preview,
        index=IntSlider(min=0, max=len(frames) - 1, value=0, description="frame"),
        threshold=FloatSlider(min=0.05, max=0.95, step=0.05, value=0.3, description="conf"),
    )
except ImportError:
    print("pip install ipywidgets for the sliders")

## The frames it was most confident about

If these look right, the model works on your footage. If they look like
waves, foam or sand, you have found the coastal false-positive problem.

In [ ]:
THRESHOLD = 0.3

ranked = sorted(
    frames,
    key=lambda p: max([b.score for b in per_frame[p]], default=0.0),
    reverse=True,
)
dmd.show_grid(
    [dmd.draw_boxes(p, per_frame[p], threshold=THRESHOLD) for p in ranked[:6]],
    columns=3,
    titles=[f"{p.name}  best {max([b.score for b in per_frame[p]], default=0):.2f}"
            for p in ranked[:6]],
)
plt.show()

## And the frames it found nothing in

Open a few. If an animal is plainly there, that is your failure mode - and
the next cell tells you whether it is a size problem.

In [ ]:
blank = [p for p in frames if not [b for b in per_frame[p] if b.score >= THRESHOLD]]
print(f"{len(blank)} of {len(frames)} frames have nothing above {THRESHOLD}")

if blank:
    dmd.show_grid(
        [dmd.draw_boxes(p, per_frame[p], threshold=FLOOR) for p in blank[:6]],
        columns=3,
        titles=[f"{p.name} (showing everything >= {FLOOR})" for p in blank[:6]],
    )
    plt.show()

## Are the targets big enough?

A detector needs roughly **32 px** on the longest side. If the boxes it does
produce are mostly below that, this is a resolution problem and no amount of
fine-tuning fixes it - fly lower, or raise `MAX_EDGE`.

In [ ]:
sizes = dmd.pixels_on_target([b for boxes in per_frame.values() for b in boxes
                              if b.score >= THRESHOLD])
if sizes:
    figure, axis = plt.subplots(figsize=(7, 3.5))
    axis.hist(sizes, bins=30, color="#81C784", edgecolor="#555")
    axis.axvline(32, color="#C62828", linestyle="--", label="~32 px floor")
    axis.set_xlabel("longest side of a detection (px)")
    axis.legend()
    plt.show()
    sizes.sort()
    print(f"median {sizes[len(sizes) // 2]:.0f} px, smallest {sizes[0]:.0f}, "
          f"largest {sizes[-1]:.0f}")
else:
    print(f"nothing above {THRESHOLD} to measure. Lower it and look again.")

## The scene summary

Counts straight from the boxes - the embryo of what a coordinator would
eventually be told.

Note what it does and does not say. "Two people visible at once" is a count.
"A crowd is interfering with the animal" is a judgement, and that belongs to
a person. Keeping the summary descriptive rather than diagnostic is the
same rule the rest of the project follows.

In [ ]:
summary = dv.summarise(per_frame, fps=SAMPLE_FPS, threshold=THRESHOLD)
print(json.dumps(summary, indent=2))

print("\nin words:")
for name in DMD_CLASSES:
    if name not in summary["peak_simultaneous"]:
        continue
    print(f"  {summary['peak_simultaneous'][name]} {name}(s) at once, "
          f"present in {summary['share_of_frames'][name]:.0%} of the clip, "
          f"first seen at {summary['first_seen_seconds'][name]:.0f}s")

## Save it, and make an annotated video

`annotated.mp4` is the thing to put in a presentation.

In [ ]:
annotated = WORK / "annotated"
annotated.mkdir(parents=True, exist_ok=True)
for path in frames:
    dmd.draw_boxes(path, per_frame[path], threshold=THRESHOLD).save(
        annotated / path.name, quality=90
    )

out = dv.stitch(annotated, WORK / "annotated.mp4", SAMPLE_FPS)
print("wrote", out)

(WORK / "summary.json").write_text(json.dumps(summary, indent=2) + "\n")
print("wrote", WORK / "summary.json")

## The same thing from the shell

Everything above is one command, if you just want the outputs:

```bash
python detect_video.py MY_CLIP.MP4 --out ~/aerial-data/runs/clip1 --fps 2 --conf 0.3

# a fine-tune instead of stock DMD
python detect_video.py MY_CLIP.MP4 --out ~/aerial-data/runs/clip1-seals \
    --weights runs/seals-v1/checkpoint_best_total.pth
```

## Where to go next

| If | Then |
| --- | --- |
| It finds your animals | Label 50 frames and get a real number - **notebook 1** |
| It misses them, but the boxes it draws are sensible | Domain gap. Fine-tune - **notebook 2** |
| It misses birds specifically | Expected; that is DMD's documented gap - **notebook 3** |
| The size histogram sat under 32 px | Resolution, not weights. Fly lower or raise `MAX_EDGE` |
| It fires on waves and sand | Label those frames as background and fine-tune; negatives teach too |

One honest note to carry forward: this clip came off the memory card at full
resolution. The **live** RTMP path decodes at 640 px, where a 2 m animal at
50 m is about 15 px across. Whatever you see here is an upper bound on what
the live feed could do.